In [17]:
import pandas as pd
import numpy as np


In [18]:
df=pd.read_csv("DateFruit_Dataset.csv")
X=df.drop("Class",axis=1)
y=df["Class"]

In [19]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.2,random_state=42)

In [20]:
from sklearn.preprocessing import StandardScaler 
ss=StandardScaler()
X_train_scaled=ss.fit_transform(X_train)
X_test_scaled=ss.transform(X_test)

In [21]:
import torch
import torch.nn as nn
import torch.optim as optimiser
from torch.utils.data import DataLoader ,TensorDataset
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [22]:
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)


In [23]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [24]:
model=ANN()
crietrion=nn.CrossEntropyLoss()
optimiser=optimiser.Adam(model.parameters())


In [25]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimiser.zero_grad()
        
        outputs = model(xb)
        loss = crietrion(outputs, yb)
        loss.backward()
        optimiser.step() # params update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.8656675815582275
epoch = 2/100, loss = 1.7090777556101482
epoch = 3/100, loss = 1.5434664090474446
epoch = 4/100, loss = 1.3627925713857014
epoch = 5/100, loss = 1.1414372722307842
epoch = 6/100, loss = 0.9573843876520792
epoch = 7/100, loss = 0.8221638103326162
epoch = 8/100, loss = 0.7111588766177496
epoch = 9/100, loss = 0.6021425873041153
epoch = 10/100, loss = 0.5553841690222422
epoch = 11/100, loss = 0.5042077203591665
epoch = 12/100, loss = 0.4766589403152466
epoch = 13/100, loss = 0.42869333426157635
epoch = 14/100, loss = 0.3993088901042938
epoch = 15/100, loss = 0.37699225048224133
epoch = 16/100, loss = 0.3488686879475911
epoch = 17/100, loss = 0.3357456475496292
epoch = 18/100, loss = 0.31201288600762683
epoch = 19/100, loss = 0.30142990748087567
epoch = 20/100, loss = 0.2792420983314514
epoch = 21/100, loss = 0.270029957095782
epoch = 22/100, loss = 0.25211017082134884
epoch = 23/100, loss = 0.24565008779366812
epoch = 24/100, loss = 0.2346122811237

In [26]:
# Evaluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # [0.2, 0.5, 1.3, -0.5, ..] - 7 vals
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch

print("accuracy: ", correct/total * 100)

accuracy:  83.86648122392212


In [27]:
y_test.shape

(719,)